# Multiplexing diffusion with and without aggregates

Compare the same tracking data with no size exclusion and after removing trajectories with mean spot FWHM >0.5 µm (`SPOT_SIZE_THRESHOLD_UM`). This follows the whole-trajectory filter used in Figure S10. Size is used as an aggregate proxy.

Both runs use the original Figure 8 calibration, minimum length (10 frames), and diffusion-fit settings. Outputs go to `results_aggregate_comparison_0p5um/{results,data,figures}/`, separated by variant. Run all cells in order.

In [1]:
from pathlib import Path
import importlib
import sys

WORKING_DIR = Path.cwd().resolve()
MODULE_CANDIDATES = (
    WORKING_DIR,
    WORKING_DIR / "Figure_8_Multiplexing",
    WORKING_DIR / "notebooks" / "Figure_8_Multiplexing",
)
NOTEBOOK_DIR = next(
    (path for path in MODULE_CANDIDATES if (path / "multiplexing_msd.py").is_file()),
    None,
)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError("Could not locate multiplexing_msd.py.")
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import multiplexing_msd
multiplexing_msd = importlib.reload(multiplexing_msd)


import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
%matplotlib inline

## Settings

In [2]:
RESULTS_DIR = NOTEBOOK_DIR / "data" / "final_data"
SPOT_SIZE_THRESHOLD_UM = 0.5
SIZE_FILTER_MODE = "trajectory_mean"
THRESHOLD_TOKEN = f"{SPOT_SIZE_THRESHOLD_UM:g}".replace(".", "p")
OUTPUT_ROOT = NOTEBOOK_DIR / f"results_aggregate_comparison_{THRESHOLD_TOKEN}um"
VARIANTS = {
    "with_aggregates": {"label": "With aggregates", "threshold_um": None},
    "without_aggregates": {"label": "Without aggregates", "threshold_um": SPOT_SIZE_THRESHOLD_UM},
}
for kind in ("results", "data", "figures"):
    for variant in (*VARIANTS, "comparison"):
        (OUTPUT_ROOT / kind / variant).mkdir(parents=True, exist_ok=True)

CHANNELS = (0, 1)
CHANNEL_CONDITION_LABELS = {0: "SunTag", 1: r"UTag($\Delta$Cys)"}
SOURCE_DATA_CONDITION_LABELS = {0: "SunTag", 1: "UTag_DeltaCys"}
CHANNEL_COLORS = {0: "#509441", 1: "#ED54F8"}
MEASUREMENT_CHANNEL_MAP = {0: 0, 1: 1}
TRAJECTORY_REDUCER = "mean"
CELL_REDUCER = "mean"
MINIMUM_TRAJECTORY_FRAMES = 10
REJECT_TRAJECTORY_GAPS = False
MAXIMUM_FIT_LAG_FRAMES = 10
MAXIMUM_FIT_LAG_SECONDS = multiplexing_msd.fit_lag_seconds_from_frames(
    RESULTS_DIR, MAXIMUM_FIT_LAG_FRAMES
)
MAXIMUM_COMPUTED_LAG_FRAMES = 15
MINIMUM_CONTRIBUTORS_PER_LAG = 5
REMOVE_DRIFT = False
STRICT_VALIDATION = False

PLOT_CENTER = "mean"
PLOT_UNCERTAINTY = "sem"
MSD_SCALE = "linear"
MSD_ERROR_STYLE = "band"
MSD_DISPLAY_MAX_SECONDS = None
SHOW_INDIVIDUAL_CELL_MSD = False
SHOW_FIT_LINES = False
CONNECT_PAIRED_CELLS = False
SHOW_BOXPLOTS = True
SHOW_SAMPLE_COUNTS = False
DISPLAY_UPPER_PERCENTILE = None
PLOT_FIGSIZE = (2.81, 2.14)  # width, height in inches
CELL_METRIC_FIGSIZE = PLOT_FIGSIZE
MSD_FIGSIZE = PLOT_FIGSIZE
CELL_METRICS = ["n_trajectories", "D_um2_s", "median_trajectory_frames",
                "cell_spot_intensity_au", "cell_spot_fwhm_um", "cell_spot_snr"]
CELL_YLABELS = {
    "n_trajectories": "No. trajectories per cell",
    "D_um2_s": "D (µm²/s)",
    "median_trajectory_frames": "Trajectory length (frames)",
    "cell_spot_intensity_au": "Mean spot intensity (a.u.)",
    "cell_spot_fwhm_um": "Spot size (µm)",
    "cell_spot_snr": "Spot SNR",
}
PLOT_STYLE = {
    "font.family": "sans-serif",
    "font.sans-serif": [
        "Arial"
    ],
    "font.size": 9,
    "text.color": "black",
    "axes.labelsize": 9,
    "axes.labelcolor": "black",
    "axes.labelweight": "normal",
    "axes.titleweight": "normal",
    "axes.edgecolor": "black",
    "axes.titlesize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "xtick.color": "black",
    "ytick.color": "black",
    "legend.fontsize": 8,
    "axes.labelpad": 2,
    "xtick.major.pad": 2,
    "ytick.major.pad": 2,
    "font.weight": "normal",
    "font.style": "normal"
}
print(f"Output folder: {OUTPUT_ROOT}")
print(f"Whole-trajectory mean threshold: {SPOT_SIZE_THRESHOLD_UM:g} µm")

Output folder: /Users/nzlab-la/Desktop/utag_paper/notebooks/Figure_8_Multiplexing/results_aggregate_comparison_0p5um
Whole-trajectory mean threshold: 0.5 µm


## Diffusion analysis

Recompute MSD and diffusion fits for each variant. Paired SunTag–UTag(ΔCys) statistics use Wilcoxon tests without multiplicity adjustment, as in the source notebook.

In [3]:
analyses = {}
msd_summaries = {}
for variant, settings in VARIANTS.items():
    print(f"\n=== {settings['label']} ===")
    analysis = multiplexing_msd.analyze_final_results(
        RESULTS_DIR,
        maximum_fit_lag_seconds=MAXIMUM_FIT_LAG_SECONDS,
        maximum_computed_lag_frames=MAXIMUM_COMPUTED_LAG_FRAMES,
        minimum_trajectory_frames=MINIMUM_TRAJECTORY_FRAMES,
        reject_trajectory_gaps=REJECT_TRAJECTORY_GAPS,
        maximum_trajectory_spot_fwhm_um=settings["threshold_um"],
        size_filter_mode=SIZE_FILTER_MODE,
        channels=CHANNELS,
        minimum_contributors_per_lag=MINIMUM_CONTRIBUTORS_PER_LAG,
        remove_drift=REMOVE_DRIFT,
        channel_condition_labels=CHANNEL_CONDITION_LABELS,
        measurement_channel_map=MEASUREMENT_CHANNEL_MAP,
        trajectory_reducer=TRAJECTORY_REDUCER,
        cell_reducer=CELL_REDUCER,
        strict=STRICT_VALIDATION,
        verbose=True,
    )
    analysis["paired_channel_statistics"] = multiplexing_msd.compute_paired_channel_statistics(
        analysis, metrics=CELL_METRICS, channel_pair=CHANNELS,
        test="wilcoxon", adjust_fdr=False,
    ).drop(columns="p_value_fdr_bh")
    analyses[variant] = analysis
    result_dir = OUTPUT_ROOT / "results" / variant
    data_dir = OUTPUT_ROOT / "data" / variant
    multiplexing_msd.save_final_tables(analysis, result_dir)
    multiplexing_msd.save_figure_8_def_source_tables(
        analysis, data_dir, channels=CHANNELS,
        canonical_condition_labels=SOURCE_DATA_CONDITION_LABELS,
        normal_fit_max_lag_seconds=MAXIMUM_FIT_LAG_SECONDS,
    )
    # Full retained detections retain their original cell/particle/frame identifiers.
    analysis["tracking_data"].to_csv(data_dir / "retained_tracking_detections.csv.gz", index=False)
    (result_dir / "configuration.json").write_text(
        json.dumps(analysis["configuration"], indent=2, default=str) + "\n"
    )
    msd_summaries[variant] = multiplexing_msd.build_channel_msd_summary(
        analysis, channels=CHANNELS, paired_cells_only=True,
        require_valid_normal_fit=True, center=PLOT_CENTER, uncertainty=PLOT_UNCERTAINTY,
        normal_fit_max_lag_seconds=MAXIMUM_FIT_LAG_SECONDS,
    )
    for key, table in msd_summaries[variant].items():
        if isinstance(table, pd.DataFrame):
            table.to_csv(data_dir / f"msd_{key}.csv", index=False)
    print(f"Saved results and data for {variant}")


=== With aggregates ===
Loaded 23 valid result folders (0 invalid); channels=[0, 1].
Analyzed 46 cell/channel units. Trajectory totals by channel: {0: 569, 1: 1090}
Rejected 1277 trajectories shorter than 10 frames.
Final paired-channel table: 23 cells, 46 cell/channel rows, 23 complete channel sets.
Saved results and data for with_aggregates

=== Without aggregates ===
Loaded 23 valid result folders (0 invalid); channels=[0, 1].
Analyzed 46 cell/channel units. Trajectory totals by channel: {0: 547, 1: 1064}
Rejected 1223 trajectories shorter than 10 frames.
Rejected 48 additional trajectories with mean spot FWHM above 0.5 µm.
Final paired-channel table: 23 cells, 46 cell/channel rows, 23 complete channel sets.
Saved results and data for without_aggregates


## Filter audit and per-cell changes

Export removed particle identities and compare the same cells. Calculate diffusion changes only when both fits are valid; retain invalid fits and exclusion reasons in the CSV.

In [4]:
before = analyses["with_aggregates"]
after = analyses["without_aggregates"]
audit = after["trajectory_filter_summary"].copy()
eligible_before = before["trajectory_filter_summary"].set_index("global_particle_id")["eligible"]
audit["eligible_with_aggregates"] = audit["global_particle_id"].map(eligible_before)
assert audit["eligible_with_aggregates"].notna().all(), "The runs must contain identical source trajectories."
sizes = audit["trajectory_mean_spot_fwhm_um"]
assert np.isfinite(sizes[audit["eligible_with_aggregates"]]).all(), "Eligible trajectories have missing spot sizes."
expected_retained = audit["eligible_with_aggregates"] & sizes.le(SPOT_SIZE_THRESHOLD_UM)
assert audit["eligible"].equals(expected_retained), "Unexpected whole-trajectory filtering result."
audit["removed_as_aggregate"] = audit["eligible_with_aggregates"] & sizes.gt(SPOT_SIZE_THRESHOLD_UM)
audit.to_csv(OUTPUT_ROOT / "data/comparison/trajectory_filter_audit.csv", index=False)
audit.loc[audit.removed_as_aggregate].to_csv(
    OUTPUT_ROOT / "data/comparison/removed_aggregate_trajectories.csv", index=False)

keys = ["global_cell_id", "source_id", "local_cell_id", "channel", "channel_condition"]
comparison_columns = keys + CELL_METRICS + ["D_fit_valid", "normal_fit_exclusion_reason"]
cell_comparison = before["cell_analysis_summary"][comparison_columns].merge(
    after["cell_analysis_summary"][comparison_columns], on=keys,
    how="outer", suffixes=("_with", "_without"), validate="one_to_one", indicator=True,
)
assert cell_comparison["_merge"].eq("both").all(), "Cell/channel inventories differ."
cell_comparison = cell_comparison.drop(columns="_merge")
cell_comparison["trajectories_removed"] = (
    cell_comparison.n_trajectories_with - cell_comparison.n_trajectories_without)
cell_comparison["valid_D_in_both"] = (
    cell_comparison.D_fit_valid_with & cell_comparison.D_fit_valid_without
    & np.isfinite(cell_comparison.D_um2_s_with) & np.isfinite(cell_comparison.D_um2_s_without))
cell_comparison["D_change_without_minus_with_um2_s"] = (
    cell_comparison.D_um2_s_without - cell_comparison.D_um2_s_with
).where(cell_comparison.valid_D_in_both)
cell_comparison.to_csv(OUTPUT_ROOT / "data/comparison/paired_cell_comparison.csv", index=False)

summary_rows = []
for channel, group in cell_comparison.groupby("channel", sort=True):
    paired = group[group.valid_D_in_both]
    total = int(group.n_trajectories_with.sum())
    retained = int(group.n_trajectories_without.sum())
    summary_rows.append({
        "condition": SOURCE_DATA_CONDITION_LABELS[channel], "channel": channel,
        "n_cells": len(group), "trajectories_with_aggregates": total,
        "trajectories_without_aggregates": retained, "trajectories_removed": total - retained,
        "retained_percent": 100 * retained / total if total else np.nan,
        "valid_D_cells_with": int(group.D_fit_valid_with.sum()),
        "valid_D_cells_without": int(group.D_fit_valid_without.sum()),
        "n_cells_valid_D_in_both": len(paired),
        "mean_D_with_matched_cells_um2_s": paired.D_um2_s_with.mean(),
        "mean_D_without_matched_cells_um2_s": paired.D_um2_s_without.mean(),
        "mean_D_change_matched_cells_um2_s": paired.D_change_without_minus_with_um2_s.mean(),
    })
comparison_summary = pd.DataFrame(summary_rows)
comparison_summary.to_csv(OUTPUT_ROOT / "results/comparison/aggregate_filter_summary.csv", index=False)
display(comparison_summary)
display(cell_comparison.loc[~cell_comparison.valid_D_in_both, keys + [
    "D_fit_valid_with", "D_fit_valid_without", "normal_fit_exclusion_reason_with",
    "normal_fit_exclusion_reason_without"]])

,condition,channel,n_cells,trajectories_with_aggregates,trajectories_without_aggregates,trajectories_removed,retained_percent,valid_D_cells_with,valid_D_cells_without,n_cells_valid_D_in_both,mean_D_with_matched_cells_um2_s,mean_D_without_matched_cells_um2_s,mean_D_change_matched_cells_um2_s
0,SunTag,0,23,569,547,22,96.133568,23,23,23,0.014911,0.016044,0.001133
1,UTag_DeltaCys,1,23,1090,1064,26,97.614679,23,23,23,0.013749,0.014445,0.000696


,global_cell_id,source_id,local_cell_id,channel,channel_condition,D_fit_valid_with,D_fit_valid_without,normal_fit_exclusion_reason_with,normal_fit_exclusion_reason_without


## Percentage above the size threshold

The table follows the Figure S10 format and `SIZE_FILTER_MODE`. At the current 0.5 µm cutoff, it counts whole-trajectory means above the threshold among trajectories passing the length/gap rules before size filtering. Mean and median percentages give each cell equal weight; the pooled percentage is 100 × summed counts above threshold / summed valid counts.

In [9]:
size_filter_summary = after["trajectory_size_filter_summary"]
summary_unit = "trajectory" if SIZE_FILTER_MODE == "trajectory_mean" else "detection"
valid_units_column = (
    "n_trajectories_with_valid_spot_fwhm" if summary_unit == "trajectory"
    else "n_valid_spot_size_detections"
)
large_spot_cells = size_filter_summary[[
    "global_cell_id", "source_id", "local_cell_id", "channel", "channel_condition",
    "n_size_units_above_threshold", valid_units_column,
]].rename(columns={
    "channel_condition": "condition",
    "n_size_units_above_threshold": "units_above_threshold",
    valid_units_column: "total_valid_units",
}).copy()
large_spot_cells["threshold_um"] = SPOT_SIZE_THRESHOLD_UM
large_spot_cells["size_filter_mode"] = SIZE_FILTER_MODE
large_spot_cells["summary_unit"] = summary_unit
large_spot_cells["percentage_above_threshold"] = (
    100 * large_spot_cells["units_above_threshold"]
    / large_spot_cells["total_valid_units"].where(large_spot_cells["total_valid_units"] > 0)
)
large_spot_condition_summary = large_spot_cells.groupby("condition", sort=False).agg(
    cells=("global_cell_id", "nunique"),
    units_above_threshold=("units_above_threshold", "sum"),
    total_valid_units=("total_valid_units", "sum"),
    mean_cell_percentage=("percentage_above_threshold", "mean"),
    median_cell_percentage=("percentage_above_threshold", "median"),
).reset_index()
large_spot_condition_summary["pooled_percentage_above_threshold"] = (
    100 * large_spot_condition_summary["units_above_threshold"]
    / large_spot_condition_summary["total_valid_units"].where(
        large_spot_condition_summary["total_valid_units"] > 0)
)
large_spot_condition_summary.insert(1, "size_filter_mode", SIZE_FILTER_MODE)
large_spot_condition_summary.insert(2, "summary_unit", summary_unit)
large_spot_cells.to_csv(
    OUTPUT_ROOT / "data/comparison/large_spot_percentages_per_cell.csv", index=False)
large_spot_summary_path = OUTPUT_ROOT / "results/comparison/large_spot_condition_summary.csv"
large_spot_condition_summary.to_csv(large_spot_summary_path, index=False)
print(f"Saved: {large_spot_summary_path}")
display(large_spot_condition_summary)


Saved: /Users/nzlab-la/Desktop/utag_paper/notebooks/Figure_8_Multiplexing/results_aggregate_comparison_0p5um/results/comparison/large_spot_condition_summary.csv


,condition,size_filter_mode,summary_unit,cells,units_above_threshold,total_valid_units,mean_cell_percentage,median_cell_percentage,pooled_percentage_above_threshold
0,SunTag,trajectory_mean,trajectory,23,22,569,4.596350,0.000000,3.866432
1,UTag($\Delta$Cys),trajectory_mean,trajectory,23,26,1090,3.760613,0.925926,2.385321


## Plot helpers

The following functions retain the original stacked layout and box-and-whisker styling.

In [5]:
PLOT_DIR = None

import re
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import font_manager
from matplotlib.ticker import FormatStrFormatter, MaxNLocator, MultipleLocator

def scaled_font_sizes(scale):
    keys = (
        "font.size",
        "axes.labelsize",
        "axes.titlesize",
        "xtick.labelsize",
        "ytick.labelsize",
        "legend.fontsize",
    )
    base_points = float(mpl.rcParams["font.size"])
    sizes = {}
    for key in keys:
        value = mpl.rcParams[key]
        try:
            points = float(value)
        except (TypeError, ValueError):
            points = base_points * font_manager.font_scalings.get(str(value), 1.0)
        sizes[key] = points * scale
    return sizes

expand_svg_font_shorthand = multiplexing_msd.expand_svg_font_shorthand

def draw_metric_box_swarm(
    ax,
    analysis,
    *,
    metric,
    channels,
    statistics=None,
    statistics_label_style="stars",
    ylabel=None,
    box_width=0.55,
    whis=1.5,
    swarm_size=5.0,
    swarm_color="black",
    median_color="red",
    line_width=1.6,
    x_padding=0.62,
    bracket_height_fraction=0.10,
):
    labels = {channel: label.replace(r"$\Delta$", "Δ")
              for channel, label in analysis["configuration"]["channel_condition_labels"].items()}
    names = [str(labels[channel]) for channel in channels]
    data = multiplexing_msd._metric_valid_rows(
        analysis["cell_analysis_summary"], metric
    )
    data = data[data["channel"].isin(channels)].copy()
    if data.empty:
        raise ValueError(f"No valid cell-level values are available for {metric}.")
    data["condition"] = data["channel"].map(
        {channel: str(labels[channel]) for channel in channels}
    )

    sns.boxplot(
        data=data,
        x="condition",
        y=metric,
        order=names,
        ax=ax,
        showfliers=False,
        whis=whis,
        width=box_width,
        boxprops={"facecolor": "white", "edgecolor": "black", "linewidth": line_width},
        medianprops={"color": median_color, "linewidth": line_width + 0.6},
        whiskerprops={"color": "black", "linewidth": line_width},
        capprops={"color": "black", "linewidth": line_width},
    )
    sns.swarmplot(
        data=data,
        x="condition",
        y=metric,
        order=names,
        ax=ax,
        color=swarm_color,
        size=swarm_size,
        linewidth=0,
        zorder=3,
    )

    if metric == "n_trajectories":
        ax.yaxis.set_major_locator(MaxNLocator(nbins=7, integer=True))
    elif metric == "D_um2_s":
        ax.yaxis.set_major_locator(MultipleLocator(0.01))
        ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    else:
        ax.yaxis.set_major_locator(MaxNLocator(nbins=7, min_n_ticks=5))
    values = data[metric].to_numpy(dtype=float)
    if multiplexing_msd.METRIC_SPECS[metric].get("zero_baseline", False) and (
        values.min() >= 0
    ):
        ax.set_ylim(bottom=0)
    ax.set_xlim(-x_padding, len(names) - 1 + x_padding)
    ax.set_xlabel("")
    ax.set_ylabel(
        str(ylabel)
        if ylabel is not None
        else multiplexing_msd.METRIC_SPECS[metric]["label"]
    )
    ax.set_title("")
    ax.set_facecolor("white")
    ax.grid(False, which="both")
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(line_width)
    ax.tick_params(axis="both", which="both", colors="black")

    if statistics is not None and not statistics.empty and len(channels) >= 2:
        stat_row = statistics.loc[statistics["metric"] == metric]
        if not stat_row.empty:
            adjusted = "p_value_fdr_bh" in stat_row.columns
            p_column = "p_value_fdr_bh" if adjusted else "p_value"
            p_value = float(stat_row.iloc[0][p_column])
            stars = multiplexing_msd._significance_stars(p_value)
            label = (
                stars
                if statistics_label_style == "stars"
                else (
                    f"{stars}\n"
                    f"{multiplexing_msd._format_p_value(p_value, symbol='q' if adjusted else 'p')}; "
                    f"paired n = {int(stat_row.iloc[0]['n_paired_cells'])}"
                )
            )
            bottom, top = ax.get_ylim()
            span = top - bottom
            y = float(values.max()) + span * 0.06
            height = span * bracket_height_fraction * 0.25
            ax.plot(
                [0, 0, len(names) - 1, len(names) - 1],
                [y, y + height, y + height, y],
                color="black",
                linewidth=line_width * 0.8,
                clip_on=False,
            )
            ax.text(
                (len(names) - 1) / 2,
                y + height,
                label,
                ha="center",
                va="bottom",
                color="black",
                clip_on=False,
            )
            ax.set_ylim(top=max(top, y + height + span * 0.12))
    return data

def plot_metrics_over_msd(
    analysis,
    *,
    top_metrics=("n_trajectories", "D_um2_s"),
    channels=CHANNELS,
    center=PLOT_CENTER,
    uncertainty=PLOT_UNCERTAINTY,
    statistics=None,
    statistics_label_style="stars",
    panel_ylabels=None,
    box_width=0.55,
    whis=1.5,
    swarm_size=5.0,
    msd_error_style=MSD_ERROR_STYLE,
    msd_scale=MSD_SCALE,
    msd_show_individual_cells=SHOW_INDIVIDUAL_CELL_MSD,
    msd_show_fit_lines=SHOW_FIT_LINES,
    show_sample_counts=SHOW_SAMPLE_COUNTS,
    maximum_display_lag_seconds=MSD_DISPLAY_MAX_SECONDS,
    normal_fit_max_lag_seconds=MAXIMUM_FIT_LAG_SECONDS,
    paired_cells_only=True,
    require_valid_normal_fit=True,
    msd_integer_time_ticks=True,   # whole-second lags read better than 2.5 / 7.5
    metric_panel_width_in=3.70,   # one top boxplot panel
    metric_panel_height_in=3.45,
    metric_gap_in=1.45,           # between the top panels (holds the right y labels)
    msd_width_in=None,            # None -> span the full width of the metric row
    msd_height_in=4.10,
    block_gap_in=0.60,            # top panels -> MSD panel
    left_gutter_in=1.35,          # y tick labels + y label of the left column
    right_margin_in=0.45,
    top_margin_in=0.45,
    bottom_gutter_in=1.15,        # MSD x tick labels + 'Time lag (s)'
    font_scale=1.0,              # one type scale for every panel, relative to plot_style
    font_weight="normal",
    label_pad_pt=10.0,            # axis label -> tick labels; matplotlib's default 4 pt collides
    plot_style={
        **PLOT_STYLE, "font.size": 20, "axes.labelsize": 26, "axes.titlesize": 24,
        "xtick.labelsize": 22, "ytick.labelsize": 22, "legend.fontsize": 22,
        "axes.labelpad": 4, "xtick.major.pad": 3.5, "ytick.major.pad": 3.5,
    },
    output_directory=PLOT_DIR,
    filename_stem="normal_diffusion_metrics_over_msd",
    formats=("png", "svg", "pdf"),
    svg_expand_font_shorthand=True,  # see expand_svg_font_shorthand()
    dpi=600,
    show=True,
):
    selected_channels = [int(value) for value in channels]
    if panel_ylabels is None:
        panel_ylabels = CELL_YLABELS
    if statistics is None:
        stored_statistics = analysis.get("paired_channel_statistics")
        if isinstance(stored_statistics, pd.DataFrame):
            statistics = stored_statistics

    msd_result = multiplexing_msd.build_channel_msd_summary(
        analysis,
        channels=selected_channels,
        center=center,
        uncertainty=uncertainty,
        paired_cells_only=paired_cells_only,
        require_valid_normal_fit=require_valid_normal_fit,
        maximum_display_lag_seconds=maximum_display_lag_seconds,
        normal_fit_max_lag_seconds=normal_fit_max_lag_seconds,
    )

    n_metrics = len(top_metrics)
    metric_row_width_in = (
        n_metrics * metric_panel_width_in + (n_metrics - 1) * metric_gap_in
    )
    msd_width = metric_row_width_in if msd_width_in is None else float(msd_width_in)
    content_width_in = max(metric_row_width_in, msd_width)
    figure_width_in = left_gutter_in + content_width_in + right_margin_in
    msd_bottom_in = bottom_gutter_in
    metric_bottom_in = msd_bottom_in + msd_height_in + block_gap_in
    figure_height_in = metric_bottom_in + metric_panel_height_in + top_margin_in
    metric_left_in = left_gutter_in + (content_width_in - metric_row_width_in) / 2
    msd_left_in = left_gutter_in + (content_width_in - msd_width) / 2

    def axes_rect(x_in, y_in, w_in, h_in):
        return [
            x_in / figure_width_in,
            y_in / figure_height_in,
            w_in / figure_width_in,
            h_in / figure_height_in,
        ]

    style = dict(plot_style or {})
    style["axes.labelweight"] = font_weight
    style["axes.titleweight"] = font_weight
    with multiplexing_msd._publication_style(style):
        figure = plt.figure(
            figsize=(figure_width_in, figure_height_in), facecolor="white"
        )

        metric_axes = []
        with mpl.rc_context(scaled_font_sizes(font_scale)):
            for index, metric in enumerate(top_metrics):
                axis = figure.add_axes(
                    axes_rect(
                        metric_left_in
                        + index * (metric_panel_width_in + metric_gap_in),
                        metric_bottom_in,
                        metric_panel_width_in,
                        metric_panel_height_in,
                    )
                )
                draw_metric_box_swarm(
                    axis,
                    analysis,
                    metric=metric,
                    channels=selected_channels,
                    statistics=statistics,
                    statistics_label_style=statistics_label_style,
                    ylabel=panel_ylabels.get(metric),
                    box_width=box_width,
                    whis=whis,
                    swarm_size=swarm_size,
                )
                metric_axes.append(axis)

        with mpl.rc_context(scaled_font_sizes(font_scale)):
            msd_axis = figure.add_axes(
                axes_rect(msd_left_in, msd_bottom_in, msd_width, msd_height_in)
            )
            handles, labels = multiplexing_msd._draw_channel_msd_on_ax(
                msd_axis,
                msd_result,
                show_individual_cells=msd_show_individual_cells,
                error_style=msd_error_style,
                scale=msd_scale,
                show_sample_counts=show_sample_counts,
                show_fit_lines=msd_show_fit_lines,
            )
            msd_axis.set_title("")
            if msd_integer_time_ticks and msd_scale == "linear":
                msd_axis.xaxis.set_major_locator(
                    MaxNLocator(nbins=8, integer=True, steps=[1, 2, 5, 10])
                )
            msd_axis.legend(
                handles,
                labels,
                loc="upper left",
                ncol=1,
                frameon=False,
                handlelength=2.5,
            )

        for axis in [*metric_axes, msd_axis]:
            axis.yaxis.labelpad = float(label_pad_pt)
            axis.xaxis.labelpad = float(label_pad_pt)
        figure.canvas.draw()
        renderer = figure.canvas.get_renderer()
        for axis in [*metric_axes, msd_axis]:
            tick_boxes = [
                text.get_window_extent(renderer)
                for text in axis.get_yticklabels()
                if text.get_text()
            ]
            if not tick_boxes:
                continue
            gap_in = (
                min(box.x0 for box in tick_boxes)
                - axis.yaxis.label.get_window_extent(renderer).x1
            ) / figure.dpi
            if gap_in < 0.02:
                print(
                    f"Warning: '{axis.get_ylabel()}' sits {gap_in:.3f} in from its tick "
                    "labels; raise label_pad_pt or widen the gutter."
                )

        if output_directory is not None:
            destination = Path(output_directory).expanduser().resolve()
            destination.mkdir(parents=True, exist_ok=True)
            for extension in formats:
                figure.savefig(
                    destination / f"{filename_stem}.{extension}",
                    dpi=dpi,
                    facecolor="white",
                    edgecolor="white",
                    transparent=False,
                )
            if svg_expand_font_shorthand and "svg" in formats:
                expand_svg_font_shorthand(destination / f"{filename_stem}.svg")
            print(f"Saved {filename_stem}.{'/'.join(formats)} to {destination}")
        print(
            f"Figure canvas: {figure_width_in:.2f} x {figure_height_in:.2f} in "
            f"(aspect {figure_width_in / figure_height_in:.3f})"
        )
        if show:
            plt.show()
        else:
            plt.close(figure)
    return figure, metric_axes, msd_axis

## Individual and stacked figures for each variant

In [ ]:
for variant, analysis in analyses.items():
    print(f"\n=== {VARIANTS[variant]['label']} ===")
    PLOT_DIR = OUTPUT_ROOT / "figures" / variant
    paired_statistics = analysis["paired_channel_statistics"]
    significant_statistics = paired_statistics[paired_statistics["p_value"] < 0.05].copy()
    individual_figures = {}
    for metric in CELL_METRICS:
        figure, axis, summary = multiplexing_msd.plot_cell_metric(
            analysis,
            metric=metric,
            channels=CHANNELS,
            center=PLOT_CENTER,
            uncertainty=PLOT_UNCERTAINTY,
            connect_paired_cells=CONNECT_PAIRED_CELLS,
            show_box=SHOW_BOXPLOTS,
            log_scale=None,
            display_upper_percentile=DISPLAY_UPPER_PERCENTILE,
            statistics=significant_statistics,
            statistics_label_style="stars",
            show_sample_counts=SHOW_SAMPLE_COUNTS,
            show_title=False,
            ylabel=CELL_YLABELS[metric],
            figsize=CELL_METRIC_FIGSIZE,
            plot_style=PLOT_STYLE,
            output_directory=PLOT_DIR,
            show=True,
    )
        individual_figures[metric] = (figure, axis, summary)

    msd_figure, msd_axis, _ = multiplexing_msd.plot_channel_msd(
        analysis,
        channels=CHANNELS,
        center=PLOT_CENTER,
        uncertainty=PLOT_UNCERTAINTY,
        paired_cells_only=True,
        require_valid_normal_fit=True,
        show_individual_cells=SHOW_INDIVIDUAL_CELL_MSD,
        show_fit_lines=SHOW_FIT_LINES,
        error_style=MSD_ERROR_STYLE,
        scale=MSD_SCALE,
        maximum_display_lag_seconds=MSD_DISPLAY_MAX_SECONDS,
        normal_fit_max_lag_seconds=MAXIMUM_FIT_LAG_SECONDS,
        legend_position="top",
        figsize=MSD_FIGSIZE,
        show_sample_counts=SHOW_SAMPLE_COUNTS,
        show_title=False,
        plot_style=PLOT_STYLE,
        output_directory=PLOT_DIR,
        show=True,
    )

    stacked_figure, stacked_metric_axes, stacked_msd_axis = plot_metrics_over_msd(
        analysis,
        top_metrics=("n_trajectories", "D_um2_s"),
        statistics=significant_statistics,
        output_directory=PLOT_DIR,
        show=True,
    )
    plt.close("all")


=== With aggregates ===
Saved normal_diffusion_metrics_over_msd.png/svg/pdf to /Users/nzlab-la/Desktop/utag_paper/notebooks/Figure_8_Multiplexing/results_aggregate_comparison_0p5um/figures/with_aggregates
Figure canvas: 10.65 x 9.75 in (aspect 1.092)

=== Without aggregates ===
Saved normal_diffusion_metrics_over_msd.png/svg/pdf to /Users/nzlab-la/Desktop/utag_paper/notebooks/Figure_8_Multiplexing/results_aggregate_comparison_0p5um/figures/without_aggregates
Figure canvas: 10.65 x 9.75 in (aspect 1.092)


<Figure size 281x214 with 1 Axes>

<Figure size 281x214 with 1 Axes>

<Figure size 281x214 with 1 Axes>

<Figure size 281x214 with 1 Axes>

<Figure size 281x214 with 1 Axes>

... (7 outputs left out)

<Figure size 281x214 with 1 Axes>

<Figure size 281x214 with 1 Axes>

<Figure size 281x214 with 1 Axes>

<Figure size 1065x975 with 3 Axes>

## Spot-size distributions

Use the source notebook’s KDE curves, median markers, and tail inset for each variant. Each distribution uses the trajectories retained in that analysis.

In [ ]:
for variant, analysis in analyses.items():
    print(f"\n=== {VARIANTS[variant]['label']} ===")
    PLOT_DIR = OUTPUT_ROOT / "figures" / variant
    RAW_DATA_DIR = OUTPUT_ROOT / "data" / variant
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import seaborn as sns
    from scipy.stats import gaussian_kde

    SHOW_INSET = True
    INSET_XLIM = (0.5, 0.9)
    INSET_BOUNDS = (0.47, 0.46, 0.47, 0.36)  # [x0, y0, width, height]
    MAIN_XLIM = (0.18, 0.9)
    SAVE_DPI = 600
    SAVE_FORMATS = ("png", "svg")

    CH0_COLOR = "#509441"
    CH1_COLOR = "#ED54F8"

    channel_labels = analysis.get("configuration", {}).get(
        "channel_condition_labels", {0: "SunTag", 1: r"UTag($\Delta$Cys)"}
    )
    suntag_label = channel_labels.get(0, "SunTag")
    utag_label = channel_labels.get(1, r"UTag($\Delta$Cys)")

    DISTRIBUTION_COLORS = {
        suntag_label: CH0_COLOR,
        utag_label: CH1_COLOR,
        "SunTag": CH0_COLOR,
        r"UTag($\Delta$Cys)": CH1_COLOR,
        "UTag(ΔCys)": CH1_COLOR,
        "UTag_DeltaCys": CH1_COLOR,
        "UTag": CH1_COLOR,
        "Channel 0": CH0_COLOR,
        "Channel 1": CH1_COLOR,
    }
    ORDERED_CONDITIONS = [suntag_label, utag_label]

    sns.set_style("ticks")
    plt.rcParams.update({
        **PLOT_STYLE,
        "axes.linewidth": 0.84,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "svg.fonttype": "none",
    })

    tfs = analysis["trajectory_filter_summary"]
    valid_trajectories = tfs[tfs["eligible"]].copy()

    fig, ax = plt.subplots(figsize=PLOT_FIGSIZE, facecolor="white")
    curves = []
    observation_rows = []
    curve_rows = []

    for cond in ORDERED_CONDITIONS:
        grp = valid_trajectories[valid_trajectories["channel_condition"] == cond]
        values = grp["trajectory_mean_spot_fwhm_um"].dropna().to_numpy(dtype=float)
        values = values[(values > 0) & np.isfinite(values)]
        if values.size == 0:
            continue

        for val in values:
            observation_rows.append({"condition": cond, "spot_size_um": val})

        color = DISTRIBUTION_COLORS.get(cond, "#333333")
        kde = gaussian_kde(values, bw_method="scott")
        x_values = np.linspace(0.20, 0.88, 900)
        y_values = kde(x_values)

        ax.plot(
            x_values,
            y_values,
            color=color,
            linewidth=1.05,
            solid_capstyle="round",
            label=cond.replace(r"$\Delta$", "Δ"),
            zorder=3,
        )

        median_val = float(np.median(values))
        median_density = float(np.interp(median_val, x_values, y_values))
        ax.vlines(
            median_val,
            0,
            median_density,
            color=color,
            linewidth=0.483,
            linestyles=(0, (3, 2)),
            alpha=0.62,
            zorder=2,
        )

        curves.append((x_values, y_values, color))
        for x_val, y_val in zip(x_values, y_values):
            curve_rows.append({
                "condition": cond,
                "spot_size_um": x_val,
                "probability_density": y_val,
                "n_observations": int(values.size),
            })

    ax.set_xlabel("Spot Size (µm)", fontsize=PLOT_STYLE["axes.labelsize"], labelpad=PLOT_STYLE["axes.labelpad"], color="black")
    ax.set_ylabel("Probability Density", fontsize=PLOT_STYLE["axes.labelsize"], labelpad=PLOT_STYLE["axes.labelpad"], color="black")
    ax.tick_params(
        axis="both",
        which="major",
        labelsize=PLOT_STYLE["xtick.labelsize"],
        width=0.84,
        length=2.52,
        direction="out",
        colors="black",
    )
    ax.set_ylim(bottom=0)
    ax.set_xlim(MAIN_XLIM)
    for spine in ax.spines.values():
        spine.set_linewidth(0.84)
        spine.set_color("black")

    ax.legend(
        loc="lower right",
        bbox_to_anchor=(0.985, 0.12),
        ncol=1,
        frameon=False,
        fontsize=PLOT_STYLE["legend.fontsize"],
        handlelength=2.25,
        handletextpad=0.65,
        labelspacing=0.42,
        borderpad=0,
        borderaxespad=0,
        labelcolor="black",
    )

    if SHOW_INSET and curves:
        inset_ax = ax.inset_axes(INSET_BOUNDS)
        inset_ax.set_facecolor("white")
        inset_y_values = []

        for x_values, y_values, color in curves:
            inset_ax.plot(
                x_values,
                y_values,
                color=color,
                linewidth=0.798,
                solid_capstyle="round",
            )
            region = (x_values >= INSET_XLIM[0]) & (x_values <= INSET_XLIM[1])
            if np.any(region):
                inset_y_values.extend(y_values[region].tolist())

        inset_ax.set_xlim(INSET_XLIM)
        if inset_y_values:
            y_min, y_max = min(inset_y_values), max(inset_y_values)
            pad = 0.08 * max(y_max - y_min, y_max, 1e-9)
            inset_ax.set_ylim(y_min - pad, y_max + pad)

        inset_ax.set_yticks([])
        inset_ax.set_xticks(np.linspace(INSET_XLIM[0], INSET_XLIM[1], 3))
        inset_ax.tick_params(
            axis="x",
            labelsize=PLOT_STYLE["xtick.labelsize"],
            width=0.63,
            length=1.47,
            direction="out",
            colors="black",
        )
        for spine in inset_ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.84)
            spine.set_color("black")

    fig.tight_layout(pad=1.15)

    if PLOT_DIR is not None:
        PLOT_DIR.mkdir(parents=True, exist_ok=True)
        for ext in SAVE_FORMATS:
            out_fig = PLOT_DIR / f"figure_multiplexing_spot_size_distribution.{ext}"
            fig.savefig(out_fig, dpi=SAVE_DPI, facecolor="white", bbox_inches=None)
            if ext == "svg":
                multiplexing_msd.expand_svg_font_shorthand(out_fig)
            print(f"Saved figure: {out_fig}")

    if RAW_DATA_DIR is not None:
        RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
        obs_df = pd.DataFrame(observation_rows)
        obs_csv = RAW_DATA_DIR / "data_multiplexing_spot_size_distribution.csv"
        obs_df.to_csv(obs_csv, index=False)
        print(f"Saved observations: {obs_csv}")

        kde_df = pd.DataFrame(curve_rows)
        kde_csv = RAW_DATA_DIR / "data_multiplexing_spot_size_distribution_kde.csv"
        kde_df.to_csv(kde_csv, index=False)
        print(f"Saved KDE curves: {kde_csv}")

    plt.show()
    plt.close("all")



=== With aggregates ===
Saved figure: /Users/nzlab-la/Desktop/utag_paper/notebooks/Figure_8_Multiplexing/results_aggregate_comparison_0p5um/figures/with_aggregates/figure_multiplexing_spot_size_distribution.png
Saved figure: /Users/nzlab-la/Desktop/utag_paper/notebooks/Figure_8_Multiplexing/results_aggregate_comparison_0p5um/figures/with_aggregates/figure_multiplexing_spot_size_distribution.svg
Saved observations: /Users/nzlab-la/Desktop/utag_paper/notebooks/Figure_8_Multiplexing/results_aggregate_comparison_0p5um/data/with_aggregates/data_multiplexing_spot_size_distribution.csv
Saved KDE curves: /Users/nzlab-la/Desktop/utag_paper/notebooks/Figure_8_Multiplexing/results_aggregate_comparison_0p5um/data/with_aggregates/data_multiplexing_spot_size_distribution_kde.csv

=== Without aggregates ===
Saved figure: /Users/nzlab-la/Desktop/utag_paper/notebooks/Figure_8_Multiplexing/results_aggregate_comparison_0p5um/figures/without_aggregates/figure_multiplexing_spot_size_distribution.png
Saved

<Figure size 281x214 with 1 Axes>

<Figure size 281x214 with 1 Axes>

## Output inventory and reproducibility

In [8]:
input_paths = sorted(p for pattern in ("results_*/tracking_*.csv", "results_*/Metadata_*.txt")
                     for p in RESULTS_DIR.glob(pattern))
input_manifest = pd.DataFrame([
    {"path": str(p), "bytes": p.stat().st_size, "sha256": hashlib.sha256(p.read_bytes()).hexdigest()}
    for p in input_paths
])
input_manifest.to_csv(OUTPUT_ROOT / "data/input_manifest.csv", index=False)
run_settings = {
    "spot_size_threshold_um": SPOT_SIZE_THRESHOLD_UM, "size_filter_mode": SIZE_FILTER_MODE,
    "variants": VARIANTS, "maximum_fit_lag_frames": MAXIMUM_FIT_LAG_FRAMES,
    "maximum_fit_lag_seconds": MAXIMUM_FIT_LAG_SECONDS,
    "results_directory": str(RESULTS_DIR), "output_directory": str(OUTPUT_ROOT),
    "module_sha256": hashlib.sha256((NOTEBOOK_DIR / "multiplexing_msd.py").read_bytes()).hexdigest(),
    "python_version": sys.version, "numpy_version": np.__version__, "pandas_version": pd.__version__,
}
(OUTPUT_ROOT / "run_settings.json").write_text(json.dumps(run_settings, indent=2) + "\n")
(OUTPUT_ROOT / "README.md").write_text(
    "# Multiplexing diffusion: aggregate comparison\n\n"
    f"Whole-trajectory mean spot FWHM threshold: {SPOT_SIZE_THRESHOLD_UM:g} µm. "
    "Values above the threshold are excluded only in without_aggregates.\n\n"
    "Both variants use the same minimum trajectory length, metadata calibration, and MSD fit settings. "
    "With aggregates means no size filtering, after the shared trajectory quality rules.\n\n"
    "- results/<variant>: fit tables, exclusions, paired-channel statistics, and configuration.\n"
    "- data/<variant>: Figure 8 source tables, MSD curves, and retained detections (CSV.gz).\n"
    "- results/comparison: trajectory retention and matched-cell diffusion summaries.\n"
    "- results/comparison/large_spot_condition_summary.csv: per-condition percentages above the size threshold.\n"
    "- data/comparison/large_spot_percentages_per_cell.csv: counts and percentages for each cell.\n"
    "- data/comparison: removed particle identities, filter audit, and paired cell results.\n"
    "- figures/<variant>: whisker, KDE distribution, MSD, and stacked plots in the source notebook format.\n\n"
    "Diffusion changes are reported only for cells with valid fits in both variants. "
    
    "Paired channel p-values follow the source notebook (Wilcoxon, no multiplicity adjustment). "
    "Inspect fit-status tables before interpreting population changes.\n"
)
output_manifest = pd.DataFrame([
    {"relative_path": str(p.relative_to(OUTPUT_ROOT)), "bytes": p.stat().st_size}
    for p in sorted(OUTPUT_ROOT.rglob("*")) if p.is_file() and p.name != "output_manifest.csv"
])
output_manifest.to_csv(OUTPUT_ROOT / "output_manifest.csv", index=False)
print(f"Saved {len(output_manifest)} output files plus the manifest in {OUTPUT_ROOT}")
display(comparison_summary)

Saved 101 output files plus the manifest in /Users/nzlab-la/Desktop/utag_paper/notebooks/Figure_8_Multiplexing/results_aggregate_comparison_0p5um


,condition,channel,n_cells,trajectories_with_aggregates,trajectories_without_aggregates,trajectories_removed,retained_percent,valid_D_cells_with,valid_D_cells_without,n_cells_valid_D_in_both,mean_D_with_matched_cells_um2_s,mean_D_without_matched_cells_um2_s,mean_D_change_matched_cells_um2_s
0,SunTag,0,23,569,547,22,96.133568,23,23,23,0.014911,0.016044,0.001133
1,UTag_DeltaCys,1,23,1090,1064,26,97.614679,23,23,23,0.013749,0.014445,0.000696
